### Run this notebook online

[![Open in Colab](https://img.shields.io/badge/Open_in-Colab-F9AB00?logo=googlecolab&logoColor=F9AB00)](https://colab.research.google.com/github/hosein-fanai/continual_learning_with_diffusion_vision_transformers/blob/main/files/notebooks/hpo/DiT_Generation_HPO_Followup.ipynb)
[![Open in Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://www.kaggle.com/kernels/welcome?src=https%3A%2F%2Fgithub.com%2Fhosein-fanai%2Fcontinual_learning_with_diffusion_vision_transformers%2Fblob%2Fmain%2Ffiles%2Fnotebooks%2Fhpo%2FDiT_Generation_HPO_Followup.ipynb)

- **Colab:** click the button, choose **Runtime > Change runtime type > GPU**, and run the setup cell.
- **Kaggle:** click the button, sign in and import the notebook, then enable **Internet** and select a **GPU** in notebook settings before running setup.
- **Remote GPU container:** open this notebook inside the matching checkout with TensorFlow 2.20.0 / Keras 3.11.2 already installed. Setup verifies the environment without replacing packages.

Run the search cells individually to review the 100-trial stage before continuing.
Hosted setup uses the repository's shared installer for **TensorFlow 2.20.0 / Keras 3.11.2**;
restart the session and rerun setup if the installer requests it. CPU-only sessions
cannot run this HPO workflow.

These buttons open the **published GitHub `main` revision**. Publish this notebook,
the complete matching checkout, including the updated `common` HPO/transfer
APIs and `diffusion` model and block implementations, together before using
them. For an unpublished version, upload the notebook and that complete checkout
to the online runtime; setup reuses an existing checkout.

**Keep your results:** hosted disks can disappear when a session ends. Save the
complete `RESULTS_PATH` directory, including SQLite, sampler state, trial artifacts,
and `notebook_runner`. Restore it to the same absolute path with the same source,
Python and package versions to resume. Provider session limits still apply;
this notebook cannot keep an online runtime alive or remove its time limits.


# DiT follow-up HPO: larger models and new backbones

This separate study uses the first notebook's completed results to explore
settings that its plain-only v10 search excludes. It retains full CIFAR-10
training data, official-test feedback, 50 maximum epochs, early stopping,
MSE/MAE training with fixed EMA noise-MSE evaluation, TensorBoard and pruning.

Eight backbone families cover larger plain DiT, U-DiT, local mixers **inside**
ViT blocks, feature connections, cross-attention connections and a U-shaped
cross-attention decoder. Routed sources always precede their target stage.

The best eight distinct source conditioning/optimizer settings become partial
suggestions across the new backbone families: **88 initial suggestions** when
eight distinct hints are available. Each hint supplies four plain missing-axis
starts and one start in each of the seven other families. These suggestions
count toward the new study's attempt budget. Each new trial trains from scratch;
source scores and weights do not become target observations. TPE learns from
new outcomes. After the initial suggestions it searches the complete declared
follow-up space. The source snapshot is frozen for repeatable resumes even if
the first study later improves.

Defaults: review after 100 successful trials, main target 400, optional 50 more,
and 2,500 attempts. This notebook has its own **20-hour** budget, including two
hours for confirmation. The clock starts at its first search, not during setup
or source inspection. Trial counts are ceilings, not a promised completion time.

Run the cells in order. The configured v10 source study must contain at least one finite,
completed, compatible trial before creating the follow-up plan. It need not
have reached the first notebook's final trial target. Fewer than eight distinct
source configurations produce fewer initial suggestions. No fallback study is
selected automatically.


## Remote runtime and memory

Use a fresh remote Python kernel with TensorFlow **2.20.0** / Keras **3.11.2**.
The coordinator stays free of TensorFlow/Keras imports; admitted child processes
perform training. Never execute this notebook on the laptop.

The default is **three trials across three H100 80GB GPUs**, one per GPU, with
**73,728 MiB (72 GiB)** TensorFlow caps. Larger widths, explicit per-head key
widths, dense routes and cross-attention need substantially more memory than the
first notebook's 51-worker search. Admission adds overhead and device headroom,
checks actual device UUIDs and preserves existing jobs. This is a starting
resource allocation pending the bounded largest-case capacity check; it is not
a claim that every search-space combination fits or that three workers is the
maximum achievable concurrency.

For another runtime, change `GPU_IDS`, `CONCURRENT_TRIALS` and
`WORKER_GPU_MEMORY_LIMIT_MB` to fit its measured memory. `None` uses the API's
automatic worker cap. Each trial uses one GPU; memory is not pooled. Recognized
search OOMs are pruned. Unclassified failures remain errors, and confirmation
OOMs remain incomplete attempts.

On the supplied container, this notebook and matching APIs live in the separate
`Continual-Learning-with-Diffusion-Vision-Transformers-followup` checkout. Keep the
first notebook's original checkout intact so its source identity remains valid.
Restart the kernel when switching between these checkouts.

TensorBoard uses **port 6007** for this follow-up study.


In [ ]:
import os
import platform
import sys

from pathlib import Path
from urllib.request import urlopen


CHECKOUT_NAME = "Continual-Learning-with-Diffusion-Vision-Transformers-followup"
REPOSITORY_NAME = "continual_learning_with_diffusion_vision_transformers"
REPOSITORY = f"https://github.com/hosein-fanai/{REPOSITORY_NAME}.git"
REVISION = "main"

# Reject laptop kernels before cloning a checkout or installing any packages.
if platform.system() != "Linux":
    raise RuntimeError("Use an online GPU runtime; laptop execution is prohibited.")
if os.environ.get("KAGGLE_KERNEL_RUN_TYPE"):
    RUNTIME = "kaggle"
elif os.environ.get("COLAB_RELEASE_TAG") or "google.colab" in sys.modules:
    RUNTIME = "colab"
elif Path("/.dockerenv").is_file() or Path("/run/.containerenv").is_file():
    RUNTIME = "container"
else:
    raise RuntimeError("Use Colab, Kaggle, or a remote Linux GPU container.")

# GPU work happens only in admitted children, never in this coordinator kernel.
if "tensorflow" in sys.modules or "keras" in sys.modules:
    raise RuntimeError("Restart the kernel and run setup before importing TensorFlow or Keras.")
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"

os.environ["CONTINUAL_RUNTIME"] = RUNTIME
# Reuse the same standard-library initializer as the other online notebooks.
_locations = (
    Path.cwd(), *Path.cwd().parents, Path.cwd() / CHECKOUT_NAME, 
    Path("/kaggle/working") / CHECKOUT_NAME, Path("/content") / CHECKOUT_NAME
)
_initializer = next((
    path / "files" / "notebooks" / "init.py" for path in _locations
    if (path / "files" / "notebooks" / "init.py").is_file()
), None)
_url = (
    f"https://raw.githubusercontent.com/hosein-fanai/{REPOSITORY_NAME}/"
    f"{REVISION}/files/notebooks/init.py"
)
_setup = {"__name__": "notebook_setup", "__file__": str(_initializer or _url)}
with (_initializer.open("rb") if _initializer else urlopen(_url, timeout=30)) as _file:
    exec(compile(_file.read(), _setup["__file__"], "exec"), _setup)
CHECKOUT_ROOT, RUNTIME_PACKAGES = _setup["prepare_notebook"](
    checkout_name=CHECKOUT_NAME, 
    repository=REPOSITORY, 
    revision=REVISION, 
    runtime="local" if RUNTIME == "container" else RUNTIME, 
    cuda=False, 
    install=False if RUNTIME == "container" else None
)

os.chdir(CHECKOUT_ROOT)
if str(CHECKOUT_ROOT) not in sys.path:
    sys.path.insert(0, str(CHECKOUT_ROOT))
os.environ.setdefault("KERAS_HOME", str(CHECKOUT_ROOT.parent / ".keras"))

from common.dit_hpo_remote import inspect_remote
from common.dit_hpo_runner import (
    make_plan, search_summary, run_search, freeze_finalists, 
    run_confirmations, confirmation_summary, budget_summary
)


runtime = inspect_remote(CHECKOUT_ROOT)
display({key: value for key, value in runtime.items() if key not in {"source_sha256", "worker_policy", "concurrent_trials"}})
print("Fingerprinted source files:", len(runtime["source_sha256"]))

In [ ]:
DATASET = "CIFAR10"
SOURCE_RESULTS_PATH = "/workspace/Continual-Learning-with-Diffusion-Vision-Transformers/files/results/dit_generation_hpo_v10"
SOURCE_TOP_K = 8
RESULTS_PATH = "files/results/dit_generation_hpo_followup_v1"
VALIDATION_SOURCE = "test"
VALIDATION_RATIO = 0.0
GPU_IDS = [0, 1, 2]
CONCURRENT_TRIALS = 3
WORKER_GPU_MEMORY_LIMIT_MB = 73728
EPOCHS = 50
N_STARTUP_TRIALS = 40
SEARCH_SEED = 42

SEARCH_SPACE_OVERRIDES = {
    "dit_followup_branch": [
        "plain_missing", "u_skip", "local_hybrid", "feature_ladder", 
        "feature_dense", "cross_ladder", "cross_dense", "u_cross"
    ], 
    "followup_missing_axis": ["dim", "depth", "mha_num_heads", "mha_key_dim"], 
    "use_cfg": [True, False], 
    "patches_pos_embed_type": ["2d_sincos", "1d_sincos"], 
    "patches_pos_merger_type": ["add", "concat"], 
    "time_freq_dim": [None, 1, 2, 4, 8], 
    "time_embed_trainable": [True, False], 
    "time_mlp_ratio": [None, 1, 2, 4], 
    "label_embed_type": ["new_weight", "1d_sincos"], 
    "label_freq_dim": [None, 1, 2, 4, 8], 
    "label_mlp_ratio": [None, 1, 2, 4], 
    "dim": [16, 32, 64, 128, 256], 
    "depth": [3, 4, 5, 6, 7, 8, 9, 10], 
    "mha_num_heads": [4, 6, 8], 
    "mha_key_dim": [None, "dim"], 
    "use_refiner_cnn": [True, False], 
    "conds_merger_type": ["add", "concat"], 
    "ln_mlp_ratio": [None, 1, 2, 4], 
    "final_activation_func": ["linear", "tanh"], 
    "loss_function": ["mse", "mae"], 
    "timesteps": [500, 1000, 2500, 5000], 
    "batch_size": [32, 64, 128], 
    "local_mixer_variant": ["depthwise", "separable", "expanded_pointwise"], 
    "local_mixer_kernel_size": [3, 5, 7], 
    "local_mixer_placement": ["every", "alternating", "late"], 
    "feature_merge": ["add", "concat"], 
    "cross_merge": ["add", "concat"], 
    "cross_plug_type": ["values", "queries"]
}

PRUNING = {
    "type": "percentile", "monitor": "val_noise_loss", "percentile": 75.0, 
    "n_startup_trials": 40, "n_warmup_steps": 9, "interval_steps": 5, "n_min_trials": 10
}
REVIEW_TARGET = 100
SEARCH_TARGET = 400
EXTENSION_TARGET = SEARCH_TARGET + 50
MAX_ATTEMPTS = 2500
BATCH_TRIALS = max(100, 4 * CONCURRENT_TRIALS)
EXPERIMENT_HOURS = 20.0
CONFIRMATION_RESERVE_HOURS = 2.0
TOP_K = 3
CONFIRMATION_SEEDS = [101, 202, 303]


## Freeze the first study's results

`SOURCE_RESULTS_PATH` points to the original notebook's **v10** results. A finite,
compatible **COMPLETE** source trial and its authenticated configuration must
exist before this cell can create a transfer manifest. An unstarted or empty
study cannot start this follow-up; there is no automatic fallback to another
study. The source campaign itself need not have finished all of its targets.

The shared `freeze_transfer` API reads source artifacts without changing that
study. It ranks verified completions, keeps up to eight distinct sets of
conditioning/training hints, and freezes `source_transfer.json` under this new
results root. Once frozen, reruns reuse that snapshot even if the source changes.
Source scores rank hints but are never inserted as new objective observations;
source weights are not loaded. Width, depth, batch size, patch size and new
backbone connections remain available for fresh suggestions.

On Colab or Kaggle, restore the source artifacts at their recorded absolute
paths, or bring an already frozen `source_transfer.json` while retaining its
recorded source path. Run the next cell to create the separate follow-up plan.


In [ ]:
from common.dit_hpo_transfer import freeze_transfer


TRANSFER_MANIFEST_PATH = Path(CHECKOUT_ROOT) / RESULTS_PATH / "source_transfer.json"
transfer = freeze_transfer(
    SOURCE_RESULTS_PATH, TRANSFER_MANIFEST_PATH, top_k=SOURCE_TOP_K
)
display([
    {"source_trial": row["trial_number"], "source_loss": row["value"]}
    for row in transfer["selected"]
])
print("Frozen source manifest:", TRANSFER_MANIFEST_PATH)


In [ ]:
plan = make_plan(
    CHECKOUT_ROOT, RESULTS_PATH, dataset_name=DATASET, epochs=EPOCHS, 
    n_startup_trials=N_STARTUP_TRIALS, concurrent_trials=CONCURRENT_TRIALS, 
    gpu_ids=GPU_IDS, pruning=PRUNING, validation_source=VALIDATION_SOURCE, 
    validation_ratio=VALIDATION_RATIO, experiment_hours=EXPERIMENT_HOURS, 
    confirmation_reserve_hours=CONFIRMATION_RESERVE_HOURS, 
    search_space_overrides=SEARCH_SPACE_OVERRIDES, 
    worker_gpu_memory_limit_mb=WORKER_GPU_MEMORY_LIMIT_MB, 
    transfer_manifest=transfer, seed=SEARCH_SEED
)
TENSORBOARD_DIRECTORY = Path(plan["study_root"])
print("New study:", plan["study_root"])
print("Queued parameter suggestions:", len(plan["hpo"]["initial_trials"]))
print("TensorBoard root:", TENSORBOARD_DIRECTORY)
display(plan["hpo"])
display(budget_summary(plan))


## Search-trial pruning and memory exhaustion

The shared HPO API uses Optuna's **PercentilePruner** on **val_noise_loss**,
the existing validation EMA noise MSE. A percentile of **75** retains the better
75% of reference performance, providing a conservative screen for poor settings.
A trial's best reported value so far is compared with completed trials' values
at the same epoch; a bad single epoch does not by itself define an outlier.

Pruning begins only after **40 completed reference trials**, no earlier than
**epoch 10**, and is checked every **5 epochs** thereafter. At least **10 completed
reference trials** must have reported that epoch. These are pruning safeguards;
they do not add trials to the 100 -> 400 -> 450 search targets. NaN/Inf loss guards
remain separate and stop numerical divergence without waiting for warmup.

Workers report epoch metrics to the single CPU coordinator. Only that coordinator
calls Optuna's report/prune APIs and writes the study database, including during
multi-GPU search. Pruned trials retain intermediate values, reason/evidence, and
TensorBoard metrics, release their GPU slot, and are recorded as **PRUNED**.
They count toward **MAX_ATTEMPTS=2500**, but not toward the **450 successful-trial ceiling**.
The target is therefore not guaranteed to fit within the attempt ceiling.

**Out-of-memory handling is separate from performance pruning.** A recognized
TensorFlow `ResourceExhaustedError` or Python `MemoryError` records the search
trial as **PRUNED** for resource exhaustion. This applies even with
**PRUNING = None** and does not wait for the startup or epoch warmup. The
original exception remains in trial metadata and worker logs, and TensorBoard
receives an OOM diagnostic. Any epoch metrics already written remain available.

The affected worker exits and releases its reservation; the scheduler can
allocate the next trial in a fresh worker while the attempt budget remains.
An OOM-pruned trial consumes one of the 2,500 attempts and does not satisfy the
450 finite-COMPLETE ceiling. The engine does not silently change the failed
trial's model or batch settings. A killed process or unexplained worker crash
without explicit OOM evidence remains an error.

The runner schedules nine fresh-seed confirmation runs within the remaining time, without
performance pruning. Their original early stopping remains enabled. Set
**PRUNING = None** to disable performance pruning in a new study; changing the
policy requires a fresh results directory because it affects candidate selection.
Confirmations are separate runs outside Optuna search-trial pruning; an OOM
during confirmation remains a failed confirmation that needs attention.

See the [Optuna PercentilePruner documentation](https://optuna.readthedocs.io/en/v4.9.0/reference/generated/optuna.pruners.PercentilePruner.html).


## Live TensorBoard

This uses the same TensorBoard notebook extension as
`13_CIFAR10_Joint_HPO.ipynb`. Open it before starting a search stage so metrics
remain visible while its workers run. The dashboard includes the complete study
and confirmation tree beneath **TENSORBOARD_DIRECTORY**.

The existing `common.train.train_model` TensorBoard callback writes epoch metrics
and hyperparameter text. Fresh and resumed search trials now share
`study_root/tensorboard`; the existing HPO outcome writer adds final validation
objectives, trial status, parameters, duration and available divergence evidence.
Resource-pruned search trials also retain the OOM reason and exception
diagnostics through this shared outcome writer.
Each confirmation attempt writes its own epoch metrics and hyperparameter text
under `notebook_runner/confirmations/.../tensorboard`; final confirmation scores
remain in the comparison table and saved receipts.

No custom training callback or duplicate HPO event writer is defined in this
notebook. The dashboard starts empty until trials create events. Hosted runtime
support controls how its embedded view or port 6007 is exposed.


In [ ]:
from IPython import get_ipython


TENSORBOARD_DIRECTORY.mkdir(parents=True, exist_ok=True)
get_ipython().run_line_magic("load_ext", "tensorboard")
get_ipython().run_line_magic(
    "tensorboard", 
    f'--logdir "{TENSORBOARD_DIRECTORY}" --port 6007 --bind_all'
)

## Search space and backbone connections

`SEARCH_SPACE_OVERRIDES` is passed to the maintained `common.hpo` API. This
notebook has no independent sampler. The new `dit_followup_branch` switch
activates the shared follow-up templates; conditional options are sampled only
for branches that use them.

| Capacity | Choices |
| --- | --- |
| Base width | `dim`: 16, 32, 64, 128, **256** |
| Processing depth | `depth`: 3, 4, 5, 6, **7, 8, 9, 10**; U backbones use a fixed nine-stage graph |
| Attention heads | `mha_num_heads`: 4, **6, 8** |
| Per-head key width | `mha_key_dim`: None or **"dim"**, resolving to the sampled base width |
| Batches | `batch_size`: **32, 64, 128** |

`None` key width uses the attention layer's native inferred width. `"dim"`
assigns the full sampled base width to **each head**, so it can be substantially
more expensive. The `plain_missing` family first selects an excluded axis:
width 256, depth 7–10, heads 6/8, or explicit per-head `"dim"`. Its remaining
axes keep their full domains. Consequently every plain candidate lies outside
the first notebook's small plain-DiT domain. The other seven families introduce
new topologies and can use any of the capacity choices above.

Depth IDs refer to stage outputs: 0 is the initial patch embedding, and a route
at stage `t` can read only earlier outputs. The templates use the following
connections; `t` starts at 3 for routed same-grid families.

| Backbone family | Computation and connections |
| --- | --- |
| `plain_missing` | Plain ViT stack with at least one formerly excluded capacity setting |
| `u_skip` | U-DiT, fixed depth 9: ViT stages 1/3/5/7/9, downsample 2/4, upsample 6/8; feature skips `7:[3,6]`, `9:[1,8]` |
| `local_hybrid` | Spatial local mixers **inside selected ViT blocks**, after attention and before their FFN |
| `feature_ladder` | Stage `t` fuses the first block and current stream: `[1,t-1]` |
| `feature_dense` | Stage `t` fuses all earlier block outputs: `[1,...,t-1]` |
| `cross_ladder` | Decoder block at `t` retains self-attention, then cross-attends to output `[t-2]` |
| `cross_dense` | Decoder block at `t` retains self-attention, then cross-attends to `[1,...,t-2]` |
| `u_cross` | U-DiT feature skips plus decoder cross-attention at `7:[3]` and `9:[1]` |

| Conditional topology choice | Choices |
| --- | --- |
| Local convolution | `depthwise`; `separable` depthwise plus pointwise; `expanded_pointwise` with twice-width pointwise channels projected back |
| Local kernel | 3, 5, 7 |
| Local placement | `every`; `alternating` stages 1/3/5/...; `late` stages after the midpoint |
| Feature merge | add or concat for feature ladder/dense routes |
| Cross-attention source merge | add or concat |
| Cross-attention side | Earlier features supplied as `values` or `queries` |
| U sampling positions | 2d_sincos or new_weight |

Native local mixers preserve the token grid, bypass any prefix tokens, and use
adaptive normalization and zero initialization. Internal channel expansion is
projected back to the block width before the FFN. Feature and cross-source
merges use compatible spatial grids; concatenated sources are projected back
to the model width. Additive routes require matching source widths. These are
native model APIs, including the optional in-block mixer extension.

| Remaining group | Parameters and choices |
| --- | --- |
| CFG support | `use_cfg`: True/False |
| Patch positions | `patches_pos_embed_type`: 2d_sincos/1d_sincos |
| Patch-position merge | `patches_pos_merger_type`: add/concat |
| Time frequencies | `time_freq_dim`: None/1/2/4/8 |
| Time trainability | `time_embed_trainable`: True/False |
| Time MLP | `time_mlp_ratio`: None/1/2/4, when frequency width is explicit |
| Label embedding | `label_embed_type`: new_weight/1d_sincos |
| Label frequencies | `label_freq_dim`: None/1/2/4/8 |
| Label MLP | `label_mlp_ratio`: None/1/2/4, when frequency width is explicit |
| Refiner CNN | `use_refiner_cnn`: True/False |
| Condition merge | `conds_merger_type`: add/concat |
| Adaptive-normalization MLP | `ln_mlp_ratio`: None/1/2/4 |
| Output activation | `final_activation_func`: linear/tanh |
| Compile loss | `loss_function`: mse/mae |
| Patch extraction | size 2/4; CNN patchification off/on |
| Transformer | FFN MLP ratio 2/4; drop-path 0/0.05/0.1 |
| Optimizer | Adam/AdamW; learning rate 3e-4 to 5e-3, log-uniform |
| AdamW only | weight decay 1e-6 to 1e-3, log-uniform |
| Clipping | per-variable None/0.5/1/5; global None/0.5/1/5 only when per-variable is None |
| Diffusion | **500/1000/2500/5000 timesteps**; clipped_cosine/squaredcos_cap_v2/linear |
| CFG label dropout | p_uncond 0.05/0.1/0.2 when CFG is enabled; otherwise zero |
| EMA | decay 0.995/0.999 |
| Auxiliary image loss | coefficient 0/0.01/0.05/0.1 |

`2d_sincos` is the model API spelling. Time/label MLP ratios are active only
when their frequency width is explicit; there None and 1 both resolve to a
one-times hidden width. For adaptive normalization, `ln_mlp_ratio=None` omits
the hidden Dense layer and 1/2/4 select its width multiplier.

Training uses the sampled MSE or MAE compile loss. Validation feedback remains
**EMA noise MSE** through the shared fixed evaluation-loss API, so early
stopping, pruning and HPO compare the same metric across training losses.

Ordinary-fit cosine learning-rate decay and EMA evaluation remain enabled.
Wrapper sampling uses 50 steps, eta 0, and CFG 4 when enabled; final image reports
use CFG 3. With CFG disabled both scales are 1. These sampling settings are fixed.
There is no distillation, reconstruction-mode override or continual replay.


## HPO engine, TensorBoard and recovery

Each stage calls **run_search**, which invokes the existing **common.hpo.run_hpo**
API with the printed settings, including `search_space_overrides`, `concurrent_trials`, `worker_gpu_ids`, `pruning`, and the verified
`worker_gpu_memory_limit_mb`. Its shared process scheduler allocates trials,
saves YAML configurations, launches **common.hpo_worker**, and records outcomes.
The runner supplies GPU admission through the API's worker resource context.
With one worker, the same API runs serially in an admitted child.

Each search batch allocates at most **BATCH_TRIALS** new trials (at least
**CONCURRENT_TRIALS** by default), with at most the requested
number training together. The shared API's total-budget mode bounds attempts.
The notebook counts finite COMPLETE trials separately and requests another bounded
batch when OOM pruning, numerical divergence or performance pruning leaves
the target incomplete. OOM-pruned workers release their reservations before
the scheduler starts replacement workers; unclassified crashes still surface.

The SQLite study, sampler state, study specification, YAML configurations,
training histories and reports persist. Re-executing a reached stage adds no
trials. Recovery uses the engine's existing study lock and checkpoint protocol.
Changing `GPU_IDS`, `CONCURRENT_TRIALS` or the worker memory cap preserves
the scientific recipe;
parallel completion order can still change adaptive suggestions.

Use a **new RESULTS_PATH** after changing the dataset, training budget, startup
budget, seed, search-space overrides, pruning policy, source implementation or validation protocol. This revision defaults
to **dit_generation_hpo_followup_v1** for the restored capacity and new backbone
choices. Keep v10 and all earlier studies intact; they cannot resume with these
different search distributions. The frozen transfer manifest and queued hint
schedule are sealed with the new recipe. Trial targets may increase without
changing the scientific recipe. Official test-set scores now guide selection.
The execution budget is saved separately in `notebook_runner/budget.json`.
At the search cutoff, completed results are kept; active trials are stopped,
recorded with deadline cancellation, and excluded from automatic recovery.
Deadline cancellation is distinct from OOM/performance pruning. Finalist
selection uses only finite COMPLETE trials. The final cutoff also stops
unfinished confirmations and preserves partial artifacts. A short cleanup
allowance is reserved before each phase boundary.


In [ ]:
display(search_summary(plan))

In [ ]:
first_stage = run_search(
    plan, 
    target_completed=REVIEW_TARGET, 
    max_attempts=MAX_ATTEMPTS, 
    batch_trials=BATCH_TRIALS
)
if not first_stage["target_reached"] and not first_stage["time_budget_exhausted"]:
    raise RuntimeError("Attempt ceiling reached before the review target; inspect failed-trial logs.")

## Review after 100 successful trials or the search deadline

Inspect failures, backbone-family and capacity coverage, best validation loss
and trial durations. The main stage then targets **400** successful trials;
the optional extension targets **450**, with at most **2,500** search attempts.
These targets include successful trials started from transferred suggestions.

The next stage preserves the data and training budget. Reached targets add no
new trials on rerun. If the search time has expired, proceed to finalist selection
using completed trials instead of trying to reach the numeric ceiling.


In [ ]:
import optuna


study = optuna.load_study(
    study_name=plan["study_name"], 
    storage="sqlite:///" + (Path(plan["study_root"]) / "study.db").as_posix()
)
trial_table = study.trials_dataframe()
display(trial_table.sort_values("value").head(15))
display(search_summary(plan))

In [ ]:
main_stage = run_search(
    plan, 
    target_completed=SEARCH_TARGET, 
    max_attempts=MAX_ATTEMPTS, 
    batch_trials=BATCH_TRIALS
)
if not main_stage["target_reached"] and not main_stage["time_budget_exhausted"]:
    raise RuntimeError("Attempt ceiling reached before the main target; inspect the search status.")

## Optional extension

`EXTENSION_TARGET = SEARCH_TARGET + 50` adds 50 successful trials when time
remains. Set it equal to `SEARCH_TARGET` to skip the extension. Increasing a
trial target does not restart or extend the persistent campaign clock.

Best-loss improvement and branch coverage are descriptive evidence, not proof
of convergence. Freeze finalists only after search workers have exited.


In [ ]:
if EXTENSION_TARGET < SEARCH_TARGET:
    raise ValueError("EXTENSION_TARGET must preserve or increase SEARCH_TARGET.")
final_stage = run_search(
    plan, 
    target_completed=EXTENSION_TARGET, 
    max_attempts=MAX_ATTEMPTS, 
    batch_trials=BATCH_TRIALS
)
if not final_stage["target_reached"] and not final_stage["time_budget_exhausted"]:
    raise RuntimeError("Resolve the incomplete search target before selecting finalists.")
display(final_stage)

## Freeze and confirm three finalists

The next cell freezes the three best distinct configurations, copies their
**original input configurations**, and records hashes and the paired seed list.
After selection is frozen this runner will not extend the search.

Confirmation uses public load_config/get_datasets/get_model/train_model/report
APIs. It preserves the original validation source and dataset shuffle stream
using SEARCH_SEED: the full official training set and official test-set validation
for this notebook. It then initializes a new model with each fresh training seed.
It never restores the candidate's trained weights. The sampled architecture,
optimizer, losses, epoch limit and early-stopping rule remain unchanged.
Search pruning and its worker communication are removed from each confirmation.
These reruns are outside Optuna search-trial pruning: confirmation OOMs remain
failed attempts and must be resolved before selecting from complete seed means.

The same three seeds are used for every finalist. Training and final stochastic
evaluation vary by confirmation seed; this is not a decomposition of individual
noise sources. Completed repeats are authenticated and skipped on rerun.
Interrupted attempts are retained and a fresh attempt directory is used.

In [ ]:
finalists = freeze_finalists(plan, CONFIRMATION_SEEDS, top_k=TOP_K)
display(finalists)

In [ ]:
confirmation_records = run_confirmations(plan)
print("Completed confirmation runs:", len(confirmation_records))

In [ ]:
import pandas as pd


summary = pd.DataFrame(confirmation_summary(plan))
summary.to_csv(Path(plan["control_root"]) / "confirmation_summary.csv", index=False)
if not summary.empty:
    complete = summary[summary["all_seeds_complete"]]
    partial = summary[~summary["all_seeds_complete"]]
    if not complete.empty:
        print("Configurations with every confirmation seed complete:")
        display(complete.sort_values("mean_noise_loss"))
    if not partial.empty:
        print("Incomplete confirmation progress (unranked). Defer final selection until all candidates finish.")
        display(partial)
else:
    print("No confirmation results are available yet.")
display(budget_summary(plan))


## Interpretation and saved artifacts

Compare mean validation noise_loss and sample standard deviation across the
three paired seeds. Inspect individual receipts and histories when candidate
differences are small. Three repeats are a practical stability check, not proof
that one configuration is universally better.

The study optimizes a denoising proxy. Image quality, diversity and downstream
replay usefulness require separate evaluation after configuration selection.
The official test set has guided tuning in this workflow. Its scores describe
selection and confirmation performance, not an untouched final evaluation.

The results root contains the frozen source_transfer.json. The study directory contains study.db, trials.csv, configs and runs.
notebook_runner contains recipe.json, search_status.json, worker requests/logs,
finalists.json, frozen YAML inputs, confirmation receipts and summary tables.
Partial/failed artifacts are retained. All notebook outputs are initially empty.
